# Resumora AI — Fine-tune DistilBERT + LoRA on Colab

This notebook is intentionally thin. It installs the repo, pulls the synthetic dataset from HF Hub, and calls `python -m training.train` with the `full` preset. The same code runs locally with the `smoke` preset.

**Before running:**
1. Switch the Colab runtime to **GPU (T4)**: Runtime -> Change runtime type -> GPU.
2. Set the `HF_USER`, `HF_DATASET_REPO`, and `HF_MODEL_REPO` variables in the next cell.
3. Run all cells top-to-bottom.

Total runtime: ~30-60 minutes on T4 with ~500 pairs and 3 epochs.

In [ ]:
HF_USER = "YOUR-USERNAME"  # set me
HF_DATASET_REPO = f"{HF_USER}/resumora-ai-dataset"
HF_MODEL_REPO = f"{HF_USER}/resumora-ai-distilbert-lora"
GITHUB_REPO_URL = "https://github.com/YOUR-USERNAME/AI-Pipeline.git"  # set me
BRANCH = "main"

## 1. Install the repo

Colab images ship with `pip` not `uv`, so we install via pip from the cloned repo.

In [ ]:
!git clone --branch {BRANCH} {GITHUB_REPO_URL} ai-pipeline
%cd ai-pipeline
!pip install -q ./packages/pipeline ./packages/training

## 2. Log in to Hugging Face

Paste your write-scope token from https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import login
login()

## 3. Pull the synthetic + gold sets from your dataset repo

In [ ]:
from pathlib import Path
from training.train.data import load_pairs_from_hub
from training.dataset.jsonl import write_pairs

synth = load_pairs_from_hub(repo_id=HF_DATASET_REPO, filename="synthetic/pairs.jsonl")
gold  = load_pairs_from_hub(repo_id=HF_DATASET_REPO, filename="gold/seed.jsonl")
Path("data/synthetic").mkdir(parents=True, exist_ok=True)
Path("data/gold").mkdir(parents=True, exist_ok=True)
write_pairs(Path("data/synthetic/pairs.jsonl"), synth)
write_pairs(Path("data/gold/seed.jsonl"), gold)
print(f"synthetic = {len(synth)} pairs, gold = {len(gold)} pairs")

## 4. Train with the `full` preset

The same CLI entry point as local dev. Watch the per-epoch validation macro-F1.

In [ ]:
!python -m training.train train --preset full --run-name colab-run-1

## 5. Evaluate against the gold set

In [ ]:
!python -m training.train evaluate --model-dir outputs/full --gold-pairs data/gold/seed.jsonl --max-length 512

## 6. Push the adapter + auto-generated model card to HF Hub

In [ ]:
import json
from pathlib import Path

# Re-run gold evaluation and persist a metrics file the publisher can embed in the card.
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from training.dataset.jsonl import read_pairs
from training.train.evaluate import evaluate_against_gold

model_dir = Path("outputs/full")
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model     = AutoModelForSequenceClassification.from_pretrained(model_dir)
gold      = read_pairs(Path("data/gold/seed.jsonl"))
report    = evaluate_against_gold(model=model, tokenizer=tokenizer, gold_pairs=gold, max_length=512, device="cuda")
(model_dir / "gold_eval.json").write_text(json.dumps(report.to_dict(), indent=2))

In [ ]:
from training.train.config import full_config
import json

train_cfg_json = json.dumps(full_config().to_dict())
!python -m training.publish.model \
    --repo {HF_MODEL_REPO} \
    --model-dir outputs/full \
    --metrics-json outputs/full/gold_eval.json \
    --dataset-repo {HF_DATASET_REPO} \
    --base-model distilbert-base-uncased \
    --train-config-json '{train_cfg_json}' \
    --message "phase 3 — initial release"

## 7. Save MLflow runs (Colab `./mlruns/` dies with the runtime)

Zip the run directory and download it so you can browse with `mlflow ui` locally.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("mlruns", "zip", ".", "mlruns")
files.download("mlruns.zip")